# Main Report Experiments

This notebook collects the experiments that support the main body of the report.
It is intended as the first notebook to inspect when reviewing the code bundle.

The original development notebooks are preserved separately for traceability. This
cleaned notebook focuses on the report-ready experiments and outputs.


## Project setup and imports

Purpose:
Establish the cleaned branch paths and point reviewers to the archived source notebooks without rerunning long training jobs by accident.

Report use:
This section is only orchestration. It should make the repository easier to inspect and make output locations explicit.

Key caveat:
Several source notebooks contain long neural-network training runs. Run individual sections deliberately rather than using "Run all" during a quick review.

Expected outputs:
No numerical outputs are expected from this setup cell. It prints the project root and key folders.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
NOTEBOOK_ARCHIVE = PROJECT_ROOT / "notebooks" / "archive" / "original_notebooks"
RESULTS_MAIN = PROJECT_ROOT / "results" / "main"
FIGURES_MAIN = PROJECT_ROOT / "figures" / "main"
HESTON_FULLINFO_RESULTS = RESULTS_MAIN / "heston_fullinfo"
HESTON_FULLINFO_FIGURES = FIGURES_MAIN / "heston_fullinfo"

# These directories are part of the cleaned branch contract. Keeping them explicit
# in the notebook makes it less likely that a rerun writes final results into an
# old ambiguous output folder.
for path in [RESULTS_MAIN, FIGURES_MAIN, HESTON_FULLINFO_RESULTS, HESTON_FULLINFO_FIGURES]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Archived source notebooks: {NOTEBOOK_ARCHIVE}")
print(f"Main results folder: {RESULTS_MAIN}")
print(f"Main figures folder: {FIGURES_MAIN}")


## Black--Scholes final benchmark

Purpose:
Compare the selected neural hedge against no hedge, Black--Scholes delta, the finite-grid discrete-time MSE-optimal hedge, and the polynomial hedge baseline.

Report use:
This is the main controlled benchmark showing that the neural architecture recovers a sensible hedge in the correctly specified Black--Scholes setting.

Key caveat:
The neural hedge is not expected to beat analytic hedges in a correctly specified Black--Scholes model. The important result is closeness to the analytic and finite-grid references.

Expected outputs:
`final_benchmark_metrics.csv`, `benchmark_rmse_bar.png`, `benchmark_hedge_error_histograms.png`, and `average_delta_by_moneyness.csv`, copied or written into `results/main/` and `figures/main/`.

Source notebook:
`notebooks/archive/original_notebooks/final_benchmark_comparison.ipynb`.


## Architecture selection or final selected architecture summary

Purpose:
Document the selected shared Markov MLP used as the Black--Scholes baseline architecture.

Report use:
The selected architecture is a three-hidden-layer, width-64, `tanh` MLP shared across hedge dates, with input `(log(S_t/K), tau/T)` and a sigmoid delta output.

Key caveat:
The network is shared across dates. It is not an RNN and not a CNN.

Expected outputs:
Architecture-selection tables and plots from the original tuning notebook, if included in the final submitted output bundle.

Source notebooks:
`ideal_minimal_task_hyperparameter_tuning_with_architectures.ipynb` and `final_benchmark_comparison.ipynb`.


## Parameter robustness under retraining

Purpose:
Retrain the selected Black--Scholes architecture across one-factor-at-a-time changes in strike, volatility, maturity, and hedge-grid size.

Report use:
Supports the claim that the selected architecture remains close to the discrete-time MSE-optimal benchmark when retrained under each scenario.

Key caveat:
This is robustness under retraining, not out-of-scenario generalization from a single fitted network.

Expected outputs:
`robustness_rmse.csv`, `robustness_compact.csv`, and robustness plots copied or written into `results/main/` and `figures/main/`.

Source notebooks:
`parameter_robustness_study.ipynb` and the robustness cells in `final_benchmark_comparison.ipynb`.


## Generalization failure of the single-scenario hedger

Purpose:
Show that a neural hedge trained only at the baseline Black--Scholes scenario can fail badly when evaluated out of scenario without retraining.

Report use:
This motivates the parameter-conditioned hedger.

Key caveat:
The failure is expected because the baseline feature construction hardcodes the training strike, omits volatility as an input, and does not train over a distribution of contracts.

Expected outputs:
`collapse_compact.csv` and any associated generalization-failure figures in `results/main/` and `figures/main/`.

Source notebook:
`final_benchmark_comparison.ipynb`.


## Parameter-conditioned hedger

Purpose:
Train a hedger over a distribution of Black--Scholes strikes and volatilities with inputs `(log(S_t/K), tau/T, K, sigma)`.

Report use:
Shows interpolation across the trained parameter region and moderate extrapolation in the tested range.

Key caveat:
The Black--Scholes analytic premium is supplied per path because a single learned scalar premium cannot price all contracts simultaneously.

Expected outputs:
`universal_robustness.csv`, `universal_robustness_with_extrapolation.csv`, `universal_heatmap.png`, and `universal_delta_by_moneyness.png`.

Source notebook:
`final_benchmark_comparison.ipynb`.


## Heston full-information stock-and-option experiment

This is the main stochastic-volatility result in the report. The neural hedge is
given direct access to the Heston variance state \(v_t\), along with the liquid
option quote and target/liquid-option moneyness features.

The observable-volatility version is a supporting robustness check and is kept in
the appendix notebook. Full-information and observable-volatility outputs must
not overwrite each other.

Purpose:
Evaluate a two-output neural hedge under Heston dynamics, using both stock and a liquid option as hedging instruments.

Report use:
This is the main Heston extension and headline stochastic-volatility result.

Key caveat:
Final Heston tables should use fair-premium centering so that hedge quality is not confused with unconditional mean gains.

Expected outputs:
Mode-safe full-information files under `results/main/heston_fullinfo/` and `figures/main/heston_fullinfo/`, especially `heston_revised_results_fair_premium_fullinfo.csv`, `heston_same_path_bs_proxy_delta_vega_fair_fullinfo.csv`, and `heston_multiseed_pairwise_improvements_fullinfo.csv`.

Source notebook:
Root-level `heston_stock_option_neural_hedging_COS_obsvol_multiseed_corrected.ipynb`; preserved copy in `notebooks/archive/original_notebooks/`. with `USE_OBSERVABLE_VOL = False`.


## Heston same-path BS-proxy diagnostic

Purpose:
Evaluate the older frozen-volatility Black--Scholes proxy Greek rule on the same COS-priced Heston option path.

Report use:
This is the strongest analytic heuristic comparison used alongside the stock-and-option NN in the Heston section.

Key caveat:
The BS-proxy Greek rule is a diagnostic heuristic, not a Heston-consistent finite-grid optimal hedge.

Expected outputs:
`heston_same_path_bs_proxy_delta_vega_fair_fullinfo.csv` and related raw-premium diagnostics in the full-information Heston folder.


## Heston full-information multi-seed headline

Purpose:
Check whether the stock-and-option NN improvement over the same-path BS-proxy Greek heuristic is stable across several independent seeds.

Report use:
Supports the conservative headline that the stock-and-option NN wins in the reported three-seed check.

Key caveat:
Three seeds are a consistency check, not a high-powered statistical estimate.

Expected outputs:
`heston_multiseed_pairwise_improvements_fullinfo.csv` and a compact summary table in `results/main/heston_fullinfo/`.


## Heston Carr--Madan validation and finite-difference Greek checks

Purpose:
Validate the COS Heston pricer against an independent Carr--Madan implementation over the traded liquid-option maturity range and record finite-difference Greek checks.

Report use:
Supports the claim that the liquid option prices used in the main Heston experiment are internally consistent and independently checked.

Key caveat:
Use conservative wording: agreement is at the `1e-8` to `1e-7` scale depending on the validation grid, not exact-agreement language.

Expected outputs:
`heston_cos_carr_madan_validation_traded_range_fullinfo.csv` and `heston_cos_greek_finite_difference_check_fullinfo.csv`.

Source notebook:
The corrected Heston notebook includes actual Carr--Madan code and finite-difference Greek checks.


In [ ]:
SOURCE_NOTEBOOKS = {
    "black_scholes_main": NOTEBOOK_ARCHIVE / "final_benchmark_comparison.ipynb",
    "architecture_selection": NOTEBOOK_ARCHIVE / "ideal_minimal_task_hyperparameter_tuning_with_architectures.ipynb",
    "parameter_robustness": NOTEBOOK_ARCHIVE / "parameter_robustness_study.ipynb",
    "heston_corrected": PROJECT_ROOT / "heston_stock_option_neural_hedging_COS_obsvol_multiseed_corrected.ipynb",
}

for label, path in SOURCE_NOTEBOOKS.items():
    print(f"{label:24s}: {path} | exists={path.exists()}")
